In [29]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/validate_submission.py
/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source2.tsv
/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source3.tsv
/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source1.tsv
/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_ground_truth.tsv
/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source3.tsv
/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source2.tsv
/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source1.tsv


In [30]:
# Cell 1 — GPU check + installs (only installs what isn't already there)
!nvidia-smi
try:
    import faiss
    print("faiss already available:", faiss.__version__)
except ImportError:
    !pip install -q faiss-gpu-cu12
    import faiss
!pip install -q sentence-transformers uroman

Sat Sep 26 08:33:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             27W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [31]:
# Cell 2 — Paths. Confirm the slug with the ls output, then set DATASET_SLUG.
import os
!ls /kaggle/input/

DATASET_SLUG = "amazon-ml-challenge-2026"   # <-- adjust to match the ls output
BASE = f"/kaggle/input/{DATASET_SLUG}/dataset"
TRAIN_DIR, TEST_DIR = f"{BASE}/train", f"{BASE}/test"

paths = {
    "s1_train": f"/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source1.tsv", "s2_train": f"/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source2.tsv",
    "s3_train": f"/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source3.tsv", "gt_train": f"/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_ground_truth.tsv",
    "s1_test": f"/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source1.tsv", "s2_test": f"/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source2.tsv",
    "s3_test": f"/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source3.tsv",
}
for k, v in paths.items():
    print(k, os.path.exists(v), v)

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

datasets
s1_train True /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source1.tsv
s2_train True /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source2.tsv
s3_train True /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source3.tsv
gt_train True /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_ground_truth.tsv
s1_test True /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source1.tsv
s2_test True /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source2.tsv
s3_test True /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source3.tsv


In [32]:
# Cell 3 — Text construction, same shape as your local tfidf_blocker.py
import uroman as ur
_uroman, _cache = None, {}

def get_uroman():
    global _uroman
    if _uroman is None:
        _uroman = ur.Uroman()
    return _uroman

def romanize_text(s: str) -> str:
    if not s or s.isascii():
        return s
    u = get_uroman()
    out = []
    for w in s.split():
        if w.isascii():
            out.append(w)
        else:
            c = _cache.get(w)
            if c is None:
                c = u.romanize_string_core(w, None, ur.RomFormat.STR, 0)
                _cache[w] = c
            out.append(c)
    return " ".join(out)

def build_text(name, addr):
    raw = f"{name or ''} {addr or ''}".lower()
    return raw if raw.isascii() else romanize_text(raw)

In [33]:
# Cell 3 — Text construction, same shape as your local tfidf_blocker.py
import uroman as ur
_uroman, _cache = None, {}

def get_uroman():
    global _uroman
    if _uroman is None:
        _uroman = ur.Uroman()
    return _uroman

def romanize_text(s: str) -> str:
    if not s or s.isascii():
        return s
    u = get_uroman()
    out = []
    for w in s.split():
        if w.isascii():
            out.append(w)
        else:
            c = _cache.get(w)
            if c is None:
                c = u.romanize_string_core(w, None, ur.RomFormat.STR, 0)
                _cache[w] = c
            out.append(c)
    return " ".join(out)

def build_text(name, addr):
    raw = f"{name or ''} {addr or ''}".lower()
    return raw if raw.isascii() else romanize_text(raw)

In [34]:
# Cell 4 — Streaming per-country reader (never holds the full file in memory)
def stream_source(path, country_filter=None):
    with open(path, "r", encoding="utf-8") as f:
        header = next(f).strip().split("\t")
        id_i, name_i = header.index("entity_id"), header.index("business_name")
        addr_i, c_i = header.index("business_address"), header.index("country")
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) <= c_i:
                continue
            country = parts[c_i].strip()
            if country_filter and country != country_filter:
                continue
            eid = parts[id_i]
            yield eid, build_text(parts[name_i] if len(parts) > name_i else "",
                                   parts[addr_i] if len(parts) > addr_i else "")

In [35]:
# Cell 5 — Load LaBSE on CPU, THEN start the pool (workers place themselves on GPU)
from sentence_transformers import SentenceTransformer
import torch

N_GPUS = torch.cuda.device_count()
print(f"GPUs visible: {N_GPUS}")

model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")  # cpu, not cuda:0
model = model.half()
EMBED_DIM = model.get_sentence_embedding_dimension()

target_devices = [f"cuda:{i}" for i in range(N_GPUS)] if N_GPUS > 1 else ["cuda:0"]
pool = model.start_multi_process_pool(target_devices=target_devices)
print("Encoding pool on:", target_devices)

GPUs visible: 2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_1111/2835532758.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = model.get_sentence_embedding_dimension()


Encoding pool on: ['cuda:0', 'cuda:1']


In [ ]:
# Cell 6 — encode via the pool; GPU index built AFTER encoding (sharded across GPUs as needed)
import faiss, numpy as np, torch
from concurrent.futures import ThreadPoolExecutor

def encode_texts(texts, batch_size=768):
    embs = model.encode(texts, pool=pool, batch_size=batch_size)
    embs = embs.astype("float32")
    faiss.normalize_L2(embs)
    return embs

def choose_num_shards(total_vecs, dim, block=500_000, gpu_budget_bytes=13 * 1024**3):
    """How many GPUs needed so no shard's build transient (old+new buffer
    during growth) exceeds a safe per-GPU budget."""
    bytes_per_vec = dim * 2  # fp16 storage
    max_gpus = torch.cuda.device_count()
    for ngpu in range(1, max_gpus + 1):
        shard_vecs = -(-total_vecs // ngpu)  # ceil division
        transient = 2 * shard_vecs * bytes_per_vec - block * bytes_per_vec
        if transient <= gpu_budget_bytes:
            return ngpu
    return max_gpus

# --- IVF tuning knobs -------------------------------------------------------
# fraction of the shard scanned per query ~= NPROBE / nlist, where
# nlist = NLIST_MULT * sqrt(shard_size). Bump NPROBE up if a recall check
# against the old exact Flat results comes back too low.
NLIST_MULT = 4
NPROBE = 32
# -----------------------------------------------------------------------------

def build_gpu_shards(all_vectors, dim, ngpu, cpu_chunk=250_000, nlist_mult=NLIST_MULT, nprobe=NPROBE):
    """Builds an IVFFlat (approximate) index per shard instead of exact Flat.
    Only ~nprobe/nlist of each shard is compared per query instead of all of it."""
    n = len(all_vectors)
    shard_size = -(-n // ngpu)
    shards = []
    for dev in range(ngpu):
        i0, i1 = dev * shard_size, min((dev + 1) * shard_size, n)
        if i0 >= i1:
            continue
        shard_n = i1 - i0
        nlist = max(1, int(nlist_mult * (shard_n ** 0.5)))

        free_before = torch.cuda.mem_get_info(dev)[0]
        print(f"[build_gpu_shards] device {dev}: {free_before/1e9:.1f}GB free before build")

        res = faiss.StandardGpuResources()
        cfg = faiss.GpuIndexIVFFlatConfig()
        cfg.useFloat16 = True
        cfg.device = dev
        gidx = faiss.GpuIndexIVFFlat(res, dim, nlist, faiss.METRIC_INNER_PRODUCT, cfg)

        # Train the coarse quantizer on a subsample -- clustering doesn't need every vector
        n_train = min(shard_n, nlist * 40)
        train_idx = np.random.choice(shard_n, n_train, replace=False)
        gidx.train(all_vectors[i0 + train_idx].astype("float32"))

        # Stage this shard's fp32 buffer in CPU-RAM-sized steps...
        fp32_buf = np.empty((shard_n, dim), dtype="float32")
        for b0 in range(0, shard_n, cpu_chunk):
            b1 = min(b0 + cpu_chunk, shard_n)
            fp32_buf[b0:b1] = all_vectors[i0 + b0:i0 + b1].astype("float32")
        gidx.add(fp32_buf)          # ...but ONE call to the GPU index
        del fp32_buf

        gidx.nprobe = nprobe
        shards.append((gidx, i0))
        print(f"[build_gpu_shards] device {dev}: {shard_n:,} vectors added, nlist={nlist}, nprobe={nprobe}")
    return shards

# One worker per shard is enough (never more shards than GPUs). faiss's search/add/train
# calls release the Python GIL, so dispatching each shard's .search() from its own thread
# actually launches the CUDA kernels concurrently instead of one-GPU-at-a-time.
_search_pool = ThreadPoolExecutor(max_workers=8)

def search_shards(shards, xq, k):
    def _search_one(shard):
        gidx, offset = shard
        s, idx = gidx.search(xq, k)
        idx = np.where(idx == -1, -1, idx + offset)
        return s, idx

    results = list(_search_pool.map(_search_one, shards))
    scores = [r[0] for r in results]
    ids = [r[1] for r in results]
    S, I = np.concatenate(scores, axis=1), np.concatenate(ids, axis=1)
    order = np.argsort(-S, axis=1)[:, :k]
    return np.take_along_axis(S, order, axis=1), np.take_along_axis(I, order, axis=1)

def free_shards(shards):
    for gidx, _ in shards:
        del gidx
    torch.cuda.empty_cache()


In [ ]:
# Cell 7 — encode candidates -> stage in CPU RAM -> free GPU -> build sharded GPU index
# -> restart pool -> encode+search queries on GPU
import time, os
from tqdm.auto import tqdm

def count_rows(path, country_filter=None):
    if country_filter is None:
        import subprocess
        return int(subprocess.check_output(["wc", "-l", path]).split()[0]) - 1
    return sum(1 for _ in stream_source(path, country_filter=country_filter))

K, CHUNK = 30, 100_000

def encode_country_candidates(country, s2_path, s3_path, cache_dir=OUTPUT_DIR):
    """Encode S2+S3 candidates for a country, caching to disk so a crash/restart
    in the indexing/search stage doesn't cost re-encoding."""
    cache_path = f"{cache_dir}/cand_cache_{country}.npz"
    tmp_cache = f"{cache_dir}/cand_cache_{country}.partial.npz"   # must already end in .npz -
                                                                   # np.savez silently appends .npz
                                                                   # otherwise, breaking the rename below

    if os.path.exists(cache_path):
        print(f"[{country}] loading cached candidates from {cache_path}")
        data = np.load(cache_path)
        return data["vecs"], data["ids"]

    if os.path.exists(tmp_cache):
        # a previous attempt finished writing but crashed/died before the rename —
        # recover it instead of re-encoding
        try:
            data = np.load(tmp_cache)
            vecs, ids = data["vecs"], data["ids"]
            os.rename(tmp_cache, cache_path)
            print(f"[{country}] recovered leftover cache {tmp_cache} -> {cache_path}")
            return vecs, ids
        except Exception as e:
            print(f"[{country}] leftover {tmp_cache} looked corrupt ({e}); re-encoding")

    t0 = time.time()
    total_cands = count_rows(s2_path, country) + count_rows(s3_path, country)
    cand_vecs = np.empty((total_cands, EMBED_DIM), dtype="float16")
    cand_ids, filled = [], 0
    pbar = tqdm(total=total_cands, desc=f"[{country}] encoding candidates")
    for src_path in (s2_path, s3_path):
        buf_ids, buf_txt = [], []
        for eid, txt in stream_source(src_path, country_filter=country):
            buf_ids.append(eid); buf_txt.append(txt)
            pbar.update(1)
            if len(buf_txt) >= CHUNK:
                embs = encode_texts(buf_txt)
                cand_vecs[filled:filled+len(embs)] = embs.astype("float16")
                filled += len(embs); cand_ids.extend(buf_ids)
                buf_ids, buf_txt = [], []
        if buf_txt:
            embs = encode_texts(buf_txt)
            cand_vecs[filled:filled+len(embs)] = embs.astype("float16")
            filled += len(embs); cand_ids.extend(buf_ids)
    pbar.close()
    cand_ids = np.array(cand_ids)
    print(f"[{country}] encoded {filled:,} candidates in {time.time()-t0:.1f}s")

    np.savez(tmp_cache, vecs=cand_vecs, ids=cand_ids)
    os.rename(tmp_cache, cache_path)
    print(f"[{country}] cached to {cache_path} ({os.path.getsize(cache_path)/1e9:.1f}GB)")
    return cand_vecs, cand_ids

def process_country(country, s1_path, s2_path, s3_path, out_path):
    global pool
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f"[{country}] {out_path} already exists — skipping")
        return

    t0 = time.time()
    cand_vecs, cand_ids = encode_country_candidates(country, s2_path, s3_path)
    filled = len(cand_vecs)

    model.stop_multi_process_pool(pool)
    torch.cuda.empty_cache()

    ngpu = choose_num_shards(filled, EMBED_DIM)
    print(f"[{country}] building on {ngpu} GPU(s)")
    shards = build_gpu_shards(cand_vecs, EMBED_DIM, ngpu)
    del cand_vecs

    pool = model.start_multi_process_pool(target_devices=target_devices)

    total_queries = count_rows(s1_path, country)
    tmp_path = out_path + ".partial"
    with open(tmp_path, "w", encoding="utf-8") as out_f:
        out_f.write("source1_entity_id\tcandidate_entity_ids\tscores\n")
        buf_ids, buf_txt = [], []
        qbar = tqdm(total=total_queries, desc=f"[{country}] querying")

        def flush():
            if not buf_txt: return
            scores, idxs = search_shards(shards, encode_texts(buf_txt), K)
            for qid, sc_row, idx_row in zip(buf_ids, scores, idxs):
                pairs = [f"{cand_ids[i]}:{s:.4f}" for i, s in zip(idx_row, sc_row) if i != -1]
                out_f.write(f"{qid}\t{';'.join(pairs)}\n")
            qbar.update(len(buf_ids))

        for eid, txt in stream_source(s1_path, country_filter=country):
            buf_ids.append(eid); buf_txt.append(txt)
            if len(buf_txt) >= CHUNK:
                flush(); buf_ids, buf_txt = [], []
        flush()
        qbar.close()

    os.rename(tmp_path, out_path)
    free_shards(shards)

    cache_path = f"{OUTPUT_DIR}/cand_cache_{country}.npz"
    if os.path.exists(cache_path):
        os.remove(cache_path)
    print(f"[{country}] total {time.time()-t0:.1f}s")


In [ ]:
# Cell 8 — Run it (unchanged in shape, but now a crash on India doesn't cost you US's completed work)
for country in ["US", "India"]:
    process_country(country, paths["s1_train"], paths["s2_train"], paths["s3_train"],
                     f"{OUTPUT_DIR}/embed_candidates_train_{country}.tsv")

In [ ]:
# Cell 9 — Same for test (adds France)
for country in ["US", "India", "France"]:
    process_country(country, paths["s1_test"], paths["s2_test"], paths["s3_test"],
                     f"{OUTPUT_DIR}/embed_candidates_test_{country}.tsv")

In [ ]:
model.stop_multi_process_pool(pool)

In [ ]:
import pandas as pd
for key in ["s2_train", "s3_train"]:
    vals = pd.read_csv(paths[key], sep="\t", usecols=["country"])
    print(key, vals["country"].value_counts().head(10))